In [ ]:
import os
import json
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

print("Library berhasil dimuat!")

In [ ]:
# Folder dataset (berada di folder yang sama dengan notebook ini)
DATASET_DIR = 'dataset/' 

# 1. Mengaktifkan Data Augmentation untuk mencegah Overfitting
datagen = ImageDataGenerator(
    rescale=1./255,          # Normalisasi piksel gambar menjadi 0-1
    rotation_range=30,       # Memutar gambar acak hingga 30 derajat
    zoom_range=0.2,          # Memperbesar/memperkecil gambar secara acak
    horizontal_flip=True,    # Membalik gambar secara horizontal
    shear_range=0.2,         # Memiringkan gambar
    fill_mode='nearest',
    validation_split=0.2     # Otomatis membagi data: 80% Training, 20% Validasi
)

# 2. Membuat train_generator (untuk belajar)
train_generator = datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(224, 224),  # Ukuran standar VGG16
    batch_size=32,
    class_mode='categorical',
    subset='training'        # Mengambil porsi 80%
)

# 3. Membuat val_generator (untuk ujian/evaluasi)
val_generator = datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'      # Mengambil porsi 20%
)

In [ ]:
# Memuat model dasar VGG16
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Bekukan (freeze) semua layer agar bobot awal tidak rusak
for layer in base_model.layers:
    layer.trainable = False

# Cairkan (unfreeze) blok konvolusi terakhir (Block 5) agar AI paham lekukan spesifik batik
for layer in base_model.layers:
    if layer.name.startswith('block5'):
        layer.trainable = True

# Membuat custom top layer
x = base_model.output
x = GlobalAveragePooling2D()(x) 
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x) # Mematikan 50% neuron acak untuk cegah overfitting

# Layer output final untuk 20 kategori motif batik
predictions = Dense(20, activation='softmax')(x) 

# Menggabungkan Model
model = Model(inputs=base_model.input, outputs=predictions)

# Compile model dengan learning rate kecil (0.0001) agar proses fine-tuning stabil
model.compile(optimizer=Adam(learning_rate=0.0001), 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

model.summary()

In [ ]:
# Rem otomatis: Berhenti jika nilai ujian (val_accuracy) tidak naik selama 5 putaran
early_stop = EarlyStopping(
    monitor='val_accuracy', 
    patience=5, 
    restore_best_weights=True
)

# Mulai melatih model
# Jangan khawatir jika berhenti sebelum epoch 50, itu artinya rem otomatis bekerja!
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=[early_stop]
)

In [ ]:
# Tentukan folder tujuan (folder models ada di direktori sebelumnya)
MODELS_DIR = '../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

# 1. Menyimpan Model AI (.h5)
MODEL_PATH = os.path.join(MODELS_DIR, 'batik_vgg16.h5')
model.save(MODEL_PATH)
print(f"Model berhasil disimpan di: {MODEL_PATH}")

# 2. Menyimpan Kamus Indeks (class_indices.json) agar web bisa membaca nama motif
LABEL_PATH = os.path.join(MODELS_DIR, 'class_indices.json')
labels = train_generator.class_indices

# Menyimpan ke dalam file JSON
with open(LABEL_PATH, 'w') as f:
    json.dump(labels, f)

print(f"File class_indices.json berhasil diperbarui di: {LABEL_PATH}")